# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [87]:
# your code here...
import re
import Bio.SeqIO
import Bio.Seq

test_cases = [
    # (read, expected result, label saying which requirement this checks)
    ("TTTTTTTTTTTTTTTTGCGGCCGCCT", "AAAAAAAAAAAAAAAA", "positive: clean read"),
    ("TTTTTTTACGTTTTTTGCGGGCCT", None, "requirement 1: upstream sequence is wrong"),
    ("TTTTTACGTTTTTTGCGGCCGCCT", None, "requirement 2: upstream sequence is not 16 bases long"),
    ("TTTTTTTTTTTXTTTTGCGGCCGCCT", None, "requirement 3: read contains invalid character"),
    ("TTTTTTttTTTTTTTTGCGGCCGCCT",  "AAAAAAAAAAAAAAAA", "requirement 4: read contains lowercase character, otherwise clean read"),
    ("AGGCGGCCGCAAAAAAAAAAAAAAAA", None, "requirement 4: read is in the incorrect orientation"),
    # YOUR TEST CASES HERE
]
#this code is writing test cases to make sure that the function we are defining works even if the inputs do not meet the requirements
#Sources: This code was written in class with the help of the instructor and my classmates Alessandro and Julia 

In [88]:
HANA_test_cases = [
# (end of NA or HA, label saying which requirement this checks)
    ("CACGATAGATAAATAATAGTGCACCAT", "positive: matches NA"),
    ("CCGGATTTGCATATAATGATGCACCAT", "positive: matches HA"),
    ("CCTTCTTTGCATATAATGATGCACCAT", "negative: matches neither HA nor NA")
]
#this code is writing test cases to ensure that the function only works when the ending sequence matches either the HA or NA end sequence 

In [89]:
def run_tests(parser_function, test_cases, HANA_test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")
    for end, label1 in HANA_test_cases: #added a for loop that runs through the HA/NA test cases as well
        result = parser_function(end)
        if result == expected:
            print(f"PASS  {label1}")
        else:
            n_failed += 1
            print(f"FAIL  {label1}: {end!r} gave {result!r}") 

    return f"{n_failed} tests failed"

#Sources: This code was given to us by the instructor, I added the second for loop to make it applicable to my HA/NA test cases as well

In [90]:

def read_barcode(seqread, bclen=16, upstream="AGGCGGCCGC", HA = "CCGGATTTGCATATAATGATGCACCAT", NA = "CACGATAGATAAATAATAGTGCACCAT"):
    """Identify a barcode with a known upstream sequence.

    Args:
        seqread: Nucleotide sequence read (a Seq object or a string) matching
            in reverse orientation.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode (forward orientation).

    Returns:
        The barcode in the forward orientation, as an upper-case string, or
        None if the read does not match the expected design.

    Example:
        >>> read_barcode(Bio.Seq.Seq("TTTTTTTTTTTTTTTTGCGGCCGCCT"), bclen=16)
        'AAAAAAAAAAAAAAAA'
    """
    seqread_upper = str(seqread).upper()
    seqread_original = str(Bio.Seq.Seq(seqread_upper).reverse_complement())

    barcode_pattern = re.compile(f"{upstream}(?P<barcode>[ATCGN]{{{bclen}}})")
    match = barcode_pattern.search(seqread_original)

    if match: 
        return match.group("barcode")
    else: 
        return None

"""this function is taking the sequence and ensuring that the barcode fits the requirements, if so, it returns the barcode only, 
if not, it returns None"""
#Sources: This code was written in class with the help of the instructor and my classmates Alessandro and Julia 

'this function is taking the sequence and ensuring that the barcode fits the requirements, if so, it returns the barcode only, \nif not, it returns None'

In [ ]:

def classify_read(seqread, upstream="AGGCGGCCGC",
                  HA="CCGGATTTGCATATAATGATGCACCAT",
                  NA="CACGATAGATAAATAATAGTGCACCAT"):
    """Return "HA", "NA", or None, based on the end sequence before the upstream marker."""
    forward = str(Bio.Seq.Seq(str(seqread).upper()).reverse_complement())
    if re.search(f"{HA}{upstream}", forward):
        return "HA"
    if re.search(f"{NA}{upstream}", forward):
        return "NA"
    return None

""" This code is defining a function that will classify each sequence as HA or NA, or return None if the input is neither HA nor NA. 
It first takes the reverse complement of the sequence, and converts it into a string. Then it checks if the sequence has the HA end or
NA end, and that this is in the correct location. 

"""

# Sources: for this code, I used Claude AI for troubleshooting to fix errors that came up, and materials from lectures 8 and 9 to write the code

' This code is defining a function that will classify each sequence as HA or NA, or return None if the input is neither HA nor NA. \nIt first takes the reverse complement of the sequence, and converts it into a string. Then it checks if the sequence has the HA end or\nNA end, and that this is in the correct location. \n\n'

In [92]:
run_tests(read_barcode, test_cases, HANA_test_cases) #this code checks my test cases for both the correct barcode and the HA or NA sequences

PASS  positive: clean read
PASS  requirement 1: upstream sequence is wrong
PASS  requirement 2: upstream sequence is not 16 bases long
PASS  requirement 3: read contains invalid character
PASS  requirement 4: read contains lowercase character, otherwise clean read
PASS  requirement 4: read is in the incorrect orientation
PASS  positive: matches NA
PASS  positive: matches HA
PASS  negative: matches neither HA nor NA


'0 tests failed'

In [ ]:
def read_barcode_info(fastq_file): 
    read_counts = {"HA": 0, "NA": 0}
    invalid_counts = {"HA": 0, "NA": 0}
    barcode_counts = {"HA": {}, "NA": {}}
    for record in Bio.SeqIO.parse(fastq_file, "fastq"):
        gene = classify_read(record.seq)
        if gene is None:
            continue                      # matches neither HA nor NA
        read_counts[gene] += 1
        barcode = read_barcode(record.seq)
        if barcode is None:
            invalid_counts[gene] += 1     # right gene, bad barcode
        else:
            barcode_counts[gene][barcode] = barcode_counts[gene].get(barcode, 0) + 1

    return read_counts, invalid_counts, barcode_counts

""" This function was created to apply the read_barcode and classify_read functions to a larger dataset, and get an output that tells us
information about the number of each type of reads. This function creates 3 dictionaries to store this information in, including the read 
counts, the invalid counts, and the barcodes themselves, along with the barcode counts within a nested dictionary. It runs each barcode
through a check of whether the gene matches HA or NA, and whether the barcode is valid. If the barcode is valid and either HA or NA matches, 
then the barcode is added to a dictionary along with the number of times that barcode appears. 

"""

read_counts, invalid_counts, barcode_counts = read_barcode_info("barcodes_R1.fastq") 
print("HA reads:", read_counts["HA"], "| NA reads:", read_counts["NA"]) 
#printing the number of NA and HA reads stored in the read_counts dictionary
# Sources: for this code, I used Claude AI for troubleshooting to fix errors that came up, and materials from lectures 8 and 9 to write the code


HA reads: 5316 | NA reads: 3973


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [94]:
# your code here...

read_counts, invalid_counts, barcode_counts = read_barcode_info("barcodes_R1.fastq")
print("Invalid HA barcodes:", invalid_counts["HA"], "| Invalid NA barcodes:", invalid_counts["NA"])
#printing the number of invalid NA and HA reads stored in the invalid_counts dictionary

Invalid HA barcodes: 66 | Invalid NA barcodes: 61


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [95]:
# your code here...
common_HA = max(barcode_counts["HA"], key=barcode_counts["HA"].get) #gives the most common barcode for HA
common_NA = max(barcode_counts["NA"], key =barcode_counts["NA"].get) #gives the most common barcode for NA

print ("The HA barcode with the most counts is",common_HA,"and the count is",barcode_counts["HA"][common_HA])
print ("The NA barcode with the most counts is",common_NA,"and the count is",barcode_counts["NA"][common_NA])

#This code pick the maximum number in the 'counts' column of the barcode_counts dictionary, and pulls out the barcode associated 
# with this number. It then uses that barcode as the key to call the number of counts 


The HA barcode with the most counts is CCCGACCCGACATTAA and the count is 155
The NA barcode with the most counts is ACCAGTTCTCCCCGGG and the count is 152
